# OpenTSLM signal probe — Colab runner

Runs the pretrained OpenTSLM HAR-CoT model under the probe's seven conditions
(400 samples each, 2800 rationales) and saves the predictions to Google Drive.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. The base LLM is gated: accept the licence of
   [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) with your
   Hugging Face account (approval can take a while).

**Timing.** About 6 s per rationale on a T4, so the full run takes about 4.5 hours.
Keep the tab open. Predictions are written to Drive after every sample and the run
resumes from them, so if the session drops, run all cells again from the top: finished
samples are skipped.

## 1. GPU

In [ ]:
!nvidia-smi -L   # must print a Tesla T4 (or another GPU)

## 2. Install OpenTSLM at the pinned commit and clone the probe

The pip resolver prints conflicts with Colab's own packages (`google-colab`, `cudf`):
they are harmless here. Colab ships `torchao` 0.10, which `peft` rejects when it loads
the LoRA weights; the probe does not need it, so it is removed.

In [ ]:
%cd /content
!test -d OpenTSLM || git clone -q https://github.com/StanfordBDHG/OpenTSLM.git
!cd OpenTSLM && git checkout -q 2968f4b && pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!test -d opentslm-signal-probe || git clone -q https://github.com/ariannabalducci/opentslm-signal-probe.git
%cd /content/opentslm-signal-probe
PROBE_REF = "main"   # a commit or tag to reproduce a past run exactly
!git fetch -q && git checkout -q {PROBE_REF} && (git pull -q 2>/dev/null || true)
!git log --oneline -1

## 3. Hugging Face login

Follow the link and enter the code it shows (or paste a read token).

In [ ]:
from huggingface_hub import login
login()

## 4. Data

Downloaded into OpenTSLM's own data folder, so OpenTSLM's dataset class finds it too.

In [ ]:
DATA = "/content/OpenTSLM/src/data/har_cot"
MODEL = "OpenTSLM/llama-3.2-1b-har-sp"
!python scripts/download_data.py --data-dir {DATA}

## 5. Google Drive

The predictions live on Drive, so a dropped session loses nothing.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/opentslm-probe"
!mkdir -p {D}
OUT = f"{D}/predictions_{MODEL.split('/')[-1]}.jsonl"
print(OUT)

## 6. Sanity check: the probe's input equals OpenTSLM's

Compares the sentences and the normalised series with OpenTSLM's own dataset class on
the whole test split. Expected: `0 text mismatches, max series difference 0.00e+00`.
**Do not continue if it reports mismatches.**

In [ ]:
!python scripts/check_against_opentslm.py --data-dir {DATA} -n 8222 --out {D}/input_check.json

## 7. Full run

One sample at a time (`--batch-size 1`), which uses OpenTSLM's own `generate`.
Batched generation gave different labels from single-sample generation on this model
(6 of 8 identical), so the script refuses `--batch-size > 1` unless `--trust-batching` is set.
The manifest written next to the predictions records the probe and OpenTSLM commits,
library versions and GPU. About 4.5 hours; re-run this cell after any
interruption and it continues where it stopped.

In [ ]:
!python scripts/run_probe.py --model {MODEL} --data-dir {DATA} --batch-size 1 --out {OUT}

## 8. Results

In [ ]:
!python scripts/analyze.py {OUT}

In [ ]:
from google.colab import files
for suffix in (".jsonl", ".summary.csv", ".analysis.json", ".manifest.json"):
    files.download(OUT.replace(".jsonl", suffix))
files.download(f"{D}/input_check.json")